# NB05 — Entrenamiento KFold de las 6 configuraciones

## Propósito

Entrenar y producir predicciones para las 6 configuraciones del trabajo:

| Configuración | Granularidad | Punto | Cabeza | Dims | n |
|---|---|---|---|---|---|
| `E_A_mlp` | Estudio | A (vistas) | MLP | 4096 | 5.000 |
| `E_A_gbm` | Estudio | A (vistas) | LightGBM | 4096 | 5.000 |
| `E_B_mlp` | Estudio | B (asimetría) | MLP | 2048 | 5.000 |
| `E_B_gbm` | Estudio | B (asimetría) | LightGBM | 2048 | 5.000 |
| `M_A_mlp` | Mama | A (CC+MLO) | MLP | 2048 | 10.000 |
| `M_A_gbm` | Mama | A (CC+MLO) | LightGBM | 2048 | 10.000 |

## Protocolo experimental

1. **Respetamos el split predefinido**: `split=test` queda intacto como hold-out limpio. Solo se evalúa al final.
2. Sobre `split=training` aplicamos **StratifiedKFold(5)** estratificando por la etiqueta `y`. Cada fold da:
   - Un entrenamiento (4 folds) + una validación (1 fold).
   - Predicciones para los estudios del fold de validación → estos van a las **predicciones OOF**.
   - Predicciones para los 1.000 estudios de test → se promediarán entre los 5 folds → **ensemble**.
3. **A nivel mama** usamos `StratifiedGroupKFold` con `groups=study_id`. Esto garantiza que **las dos mamas de un mismo paciente caen en el mismo fold**. Sin esto, la mama L de un paciente podría estar en train y la R en val, lo que inflaría el AUC artificialmente (las dos mamas comparten densidad, edad, factores genéticos, etc.).

## ¿Por qué K=5?

Compromiso clásico:
- Con K=5, cada fold de validación tiene ~800 muestras (~77 positivos a nivel estudio). Suficientes para estimar AUC con baja varianza.
- Entrenamos sobre ~3.200 (~308 positivos), que es razonable para una cabeza pequeña.
- K=10 dejaría folds más pequeños (peor estimación de AUC por fold).
- K=3 dejaría más muestras para training pero solo 3 mediciones de AUC (peor varianza estimada).

## Tres métricas que reportamos por configuración

Vamos a reportar **tres** métricas que en principio deberían ser parecidas pero pueden divergir, y la divergencia es informativa:

1. **AUC OOF**: AUC calculado sobre las predicciones out-of-fold concatenadas (todas las 4.000 predicciones, cada una hecha por el fold que NO entrenó con ese estudio).
2. **AUC media por fold ± std**: media de los 5 AUCs de validación individuales.
3. **AUC test ensemble**: AUC sobre las 1.000 predicciones de test, promediadas entre los 5 modelos.

**Cuando difieren mucho:** el MLP con BCE+pos_weight produce probabilidades mal calibradas. El sigmoide de cada fold queda "centrado" en un umbral distinto. Al concatenar OOF, los rankings entre folds se pisan y el AUC OOF baja respecto a la media de folds. **Es un artefacto conocido y reportable**: refleja descalibración, no mal rendimiento intrínseco. El AUC test ensemble (promedio de probabilidades) está menos afectado porque "promediar 5 sigmoides" suaviza el sesgo de cada uno.

In [1]:
import os, sys, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# Raíz del proyecto: por defecto, la carpeta padre de notebooks/.
# Sobrescribible con la variable de entorno TFM_PROJECT_ROOT.
BASE      = os.environ.get('TFM_PROJECT_ROOT',
                           os.path.abspath(os.path.join(os.getcwd(), '..')))
OUTPUTS      = os.path.join(BASE, 'outputs')
FEATURES_DIR = os.path.join(OUTPUTS, 'Features')
PRED_DIR     = os.path.join(OUTPUTS, 'Predicciones')
MODELS_DIR   = os.path.join(OUTPUTS, 'Models')
NB_DIR    = os.path.join(BASE, 'src')
for d in (PRED_DIR, MODELS_DIR): os.makedirs(d, exist_ok=True)

sys.path.insert(0, NB_DIR)
from tfm_models import Head, build_mlp, build_gbm   # noqa: E402  # type: ignore

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {DEVICE}')

Device: cuda


## 1. Cargar features y construir las 3 entradas

Del NB03 tenemos los vectores pooleados por vista y por par bilateral. Aquí los reorganizamos en los tres formatos que necesitamos:

- **E-A** (estudio, Punto A): aplanamos los 4 vectores de vista → matriz `(N, 4096)`.
- **E-B** (estudio, Punto B): aplanamos los 2 vectores de asimetría → matriz `(N, 2048)`.
- **M-A** (mama, Punto A): concatenamos `[L-CC, L-MLO]` y `[R-CC, R-MLO]`, después apilamos en una matriz `(2N, 2048)` con etiquetas correspondientes. **Cada estudio aporta 2 muestras** (su mama L y su mama R).

Para M-A guardamos también `groups_M` (study_id repetido) para el StratifiedGroupKFold.

In [2]:
X_view = np.load(os.path.join(FEATURES_DIR, 'X_view.npy'))   # (N, 4, 1024)
X_asym = np.load(os.path.join(FEATURES_DIR, 'X_asym.npy'))   # (N, 2, 1024)
meta   = pd.read_csv(os.path.join(FEATURES_DIR, 'metadata.csv'))

N = len(meta)
print(f'N estudios: {N}')
print(f'X_view shape: {X_view.shape}     X_asym shape: {X_asym.shape}')

# === E-A (estudio, Punto A) ===
# Aplanamos las 4 vistas en un único vector de 4096 dims
X_E_A = X_view.reshape(N, -1)              # (N, 4 * 1024) = (N, 4096)

# === E-B (estudio, Punto B) ===
X_E_B = X_asym.reshape(N, -1)              # (N, 2 * 1024) = (N, 2048)

y_E = meta['y_estudio'].values.astype(int)

# === M-A (mama, Punto A) ===
# Orden vistas en X_view: 0=L-CC, 1=R-CC, 2=L-MLO, 3=R-MLO
# Mama L = concat(L-CC, L-MLO);  Mama R = concat(R-CC, R-MLO)
X_MA_L = np.concatenate([X_view[:, 0, :], X_view[:, 2, :]], axis=1)   # (N, 2048)
X_MA_R = np.concatenate([X_view[:, 1, :], X_view[:, 3, :]], axis=1)   # (N, 2048)
X_M_A  = np.concatenate([X_MA_L, X_MA_R], axis=0)                     # (2N, 2048)
y_M    = np.concatenate([meta['y_L'].values, meta['y_R'].values]).astype(int)
# groups: las dos mamas del mismo paciente comparten study_id
groups_M = np.concatenate([meta['study_id'].values, meta['study_id'].values])
split_M  = np.concatenate([meta['split'].values, meta['split'].values])
split_E  = meta['split'].values

print(f'\n=== E-A (estudio, A) ===   X: {X_E_A.shape}   y: {y_E.shape}   pos: {y_E.sum()} ({100*y_E.mean():.2f}%)')
print(f'=== E-B (estudio, B) ===   X: {X_E_B.shape}   y: {y_E.shape}')
print(f'=== M-A (mama, A)   ===   X: {X_M_A.shape}   y: {y_M.shape}   pos: {y_M.sum()} ({100*y_M.mean():.2f}%)')
print(f'\nSplits nivel estudio: train={int((split_E=="training").sum())}  test={int((split_E=="test").sum())}')
print(f'Splits nivel mama:    train={int((split_M=="training").sum())}  test={int((split_M=="test").sum())}')

N estudios: 4999
X_view shape: (4999, 4, 1024)     X_asym shape: (4999, 2, 1024)

=== E-A (estudio, A) ===   X: (4999, 4096)   y: (4999,)   pos: 481 (9.62%)
=== E-B (estudio, B) ===   X: (4999, 2048)   y: (4999,)
=== M-A (mama, A)   ===   X: (9998, 2048)   y: (9998,)   pos: 494 (4.94%)

Splits nivel estudio: train=3999  test=1000
Splits nivel mama:    train=7998  test=2000


## 2. Función de entrenamiento de un fold — MLP

Esta función entrena un MLP con early stopping basado en AUC de validación:

1. Crea el modelo según `in_dim`.
2. Calcula `pos_weight = n_neg / n_pos` del training del fold.
3. Entrena con Adam (`lr=1e-3`, `weight_decay=1e-4`).
4. En cada epoch evalúa AUC en val. Si mejora, guarda el estado.
5. ReduceLROnPlateau baja el lr si el AUC se estanca.
6. Para cuando han pasado 15 epochs sin mejorar (`patience=15`).
7. Al final restaura el mejor estado y devuelve predicciones sobre val y opcionalmente sobre test.

**Output del modelo**: logits (la sigmoide se aplica internamente en BCEWithLogitsLoss durante entrenamiento, y manualmente con torch.sigmoid() durante predicción).

In [3]:
def train_fold_mlp(X_tr, y_tr, X_val, y_val, X_test=None,
                   max_epochs=120, batch_size=128, lr=1e-3,
                   weight_decay=1e-4, patience=15):
    """Entrena un MLP con early stopping sobre val_auc."""
    Xt  = torch.from_numpy(X_tr).float()
    yt  = torch.from_numpy(y_tr).float().unsqueeze(1)
    Xv  = torch.from_numpy(X_val).float().to(DEVICE)
    Xte = torch.from_numpy(X_test).float().to(DEVICE) if X_test is not None else None

    model = build_mlp(X_tr.shape[1]).to(DEVICE)

    # Tratamiento del desbalance: peso de la clase positiva
    n_pos = max(int(y_tr.sum()), 1)
    n_neg = max(int(len(y_tr) - y_tr.sum()), 1)
    pos_weight = torch.tensor([n_neg / n_pos], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    dl = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    best_auc, best_state, no_improve = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

        # Validación
        model.eval()
        with torch.no_grad():
            vp = torch.sigmoid(model(Xv)).cpu().numpy().ravel()
        try:
            val_auc = roc_auc_score(y_val, vp)
        except ValueError:
            val_auc = float('nan')
        scheduler.step(val_auc)

        if val_auc > best_auc:
            best_auc = val_auc; best_state = copy.deepcopy(model.state_dict()); no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    # Restaurar el mejor estado y producir predicciones finales
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        val_probs  = torch.sigmoid(model(Xv)).cpu().numpy().ravel()
        test_probs = torch.sigmoid(model(Xte)).cpu().numpy().ravel() if Xte is not None else None
    return val_probs, test_probs, best_auc, best_state

print('train_fold_mlp definida.')

train_fold_mlp definida.


## 3. Función de entrenamiento de un fold — GBM

Para LightGBM el flujo es mucho más simple:

1. Calcula `scale_pos_weight` del fold.
2. Llama a `.fit()` con `eval_set` y `early_stopping(30)`.
3. Devuelve `.predict_proba()[:, 1]` (probabilidades de clase positiva).

In [4]:
def train_fold_gbm(X_tr, y_tr, X_val, y_val, X_test=None):
    """Entrena LightGBM con early stopping sobre AUC de validación."""
    n_pos = max(int(y_tr.sum()), 1)
    n_neg = max(int(len(y_tr) - y_tr.sum()), 1)
    spw = n_neg / n_pos
    clf = build_gbm(scale_pos_weight=spw, seed=SEED)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )
    val_probs  = clf.predict_proba(X_val)[:, 1]
    test_probs = clf.predict_proba(X_test)[:, 1] if X_test is not None else None
    val_auc = roc_auc_score(y_val, val_probs)
    return val_probs, test_probs, val_auc, clf

print('train_fold_gbm definida.')

train_fold_gbm definida.


## 4. Función de KFold genérica

Esta función:
1. Separa train pool / test pool según el split predefinido.
2. Aplica StratifiedKFold (o StratifiedGroupKFold si `groups` se pasa) sobre el train pool.
3. Para cada fold: entrena la cabeza, guarda predicciones en val (van a OOF) y predicciones en test pool (luego se promedian).
4. Calcula las 3 métricas: OOF AUC, mean fold AUC, test ensemble AUC.
5. Guarda en disco las predicciones OOF y ensemble para que NB06 las pueda evaluar sin reentrenar.

In [5]:
def train_kfold(name, train_fold_fn, X, y, is_train, is_test, groups=None, n_splits=5):
    print(f'\n=== {name} ===')
    X_tr_pool = X[is_train]
    y_tr_pool = y[is_train]
    X_te_pool = X[is_test]
    y_te_pool = y[is_test]

    # Splitter: con o sin grupos
    if groups is None:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        split_iter = splitter.split(X_tr_pool, y_tr_pool)
    else:
        groups_tr = groups[is_train]
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        split_iter = splitter.split(X_tr_pool, y_tr_pool, groups=groups_tr)

    oof_preds  = np.zeros(len(y_tr_pool), dtype=np.float32)
    test_preds = np.zeros((n_splits, len(y_te_pool)), dtype=np.float32)
    fold_aucs  = []

    for fold, (tr_idx, val_idx) in enumerate(split_iter, start=1):
        t0 = time.time()
        val_probs, test_probs, val_auc, _ = train_fold_fn(
            X_tr_pool[tr_idx], y_tr_pool[tr_idx],
            X_tr_pool[val_idx], y_tr_pool[val_idx],
            X_test=X_te_pool,
        )
        oof_preds[val_idx] = val_probs
        test_preds[fold-1] = test_probs
        fold_aucs.append(val_auc)
        print(f'  fold {fold}: val_AUC={val_auc:.4f}  ({time.time()-t0:.1f}s)')

    # Las tres métricas que reportamos
    oof_auc  = roc_auc_score(y_tr_pool, oof_preds)
    test_ens = test_preds.mean(axis=0)
    test_auc = roc_auc_score(y_te_pool, test_ens)

    print(f'  → fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')
    print(f'  → mean fold AUC: {np.mean(fold_aucs):.4f}  (std {np.std(fold_aucs):.4f})')
    print(f'  → OOF AUC (training): {oof_auc:.4f}')
    print(f'  → Test AUC (hold-out, ensemble): {test_auc:.4f}')

    # Guardar predicciones para que NB06 (y luego NB07) las puedan usar
    np.save(os.path.join(PRED_DIR, f'{name}_oof.npy'),  oof_preds)
    np.save(os.path.join(PRED_DIR, f'{name}_test.npy'), test_ens)

    return {'name': name, 'oof_preds': oof_preds, 'test_preds': test_ens,
            'oof_auc': oof_auc, 'test_auc': test_auc,
            'mean_fold_auc': float(np.mean(fold_aucs)), 'std_fold_auc': float(np.std(fold_aucs))}

print('train_kfold definida.')

train_kfold definida.


## 5. Entrenar las 4 configuraciones a nivel estudio

Para nivel estudio usamos `StratifiedKFold` normal (sin groups), porque cada estudio es una muestra independiente.

In [6]:
results = {}
is_train_E = (split_E == 'training')
is_test_E  = (split_E == 'test')

results['E_A_mlp'] = train_kfold('E_A_mlp', train_fold_mlp, X_E_A, y_E, is_train_E, is_test_E)
results['E_A_gbm'] = train_kfold('E_A_gbm', train_fold_gbm, X_E_A, y_E, is_train_E, is_test_E)
results['E_B_mlp'] = train_kfold('E_B_mlp', train_fold_mlp, X_E_B, y_E, is_train_E, is_test_E)
results['E_B_gbm'] = train_kfold('E_B_gbm', train_fold_gbm, X_E_B, y_E, is_train_E, is_test_E)


=== E_A_mlp ===
  fold 1: val_AUC=0.6208  (3.1s)
  fold 2: val_AUC=0.6955  (0.8s)
  fold 3: val_AUC=0.6229  (0.9s)
  fold 4: val_AUC=0.7068  (1.5s)
  fold 5: val_AUC=0.6146  (1.9s)
  → fold AUCs: ['0.6208', '0.6955', '0.6229', '0.7068', '0.6146']
  → mean fold AUC: 0.6522  (std 0.0403)
  → OOF AUC (training): 0.5964
  → Test AUC (hold-out, ensemble): 0.6211

=== E_A_gbm ===


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 1: val_AUC=0.6298  (2.4s)


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 2: val_AUC=0.5772  (2.5s)


...

c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 3: val_AUC=0.6371  (1.0s)


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 4: val_AUC=0.6159  (0.9s)
  fold 5: val_AUC=0.6190  (0.9s)
  → fold AUCs: ['0.6286', '0.6558', '0.6371', '0.6159', '0.6190']
  → mean fold AUC: 0.6313  (std 0.0143)
  → OOF AUC (training): 0.6213
  → Test AUC (hold-out, ensemble): 0.6312


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 6. Entrenar las 2 configuraciones a nivel mama

Aquí pasamos `groups=groups_M` (study_id repetido) para que `StratifiedGroupKFold` mantenga las 2 mamas del mismo paciente juntas en el mismo fold.

In [7]:
is_train_M = (split_M == 'training')
is_test_M  = (split_M == 'test')

results['M_A_mlp'] = train_kfold('M_A_mlp', train_fold_mlp, X_M_A, y_M, is_train_M, is_test_M, groups=groups_M)
results['M_A_gbm'] = train_kfold('M_A_gbm', train_fold_gbm, X_M_A, y_M, is_train_M, is_test_M, groups=groups_M)


=== M_A_mlp ===
  fold 1: val_AUC=0.6631  (2.4s)
  fold 2: val_AUC=0.6026  (2.1s)
  fold 3: val_AUC=0.6622  (2.0s)
  fold 4: val_AUC=0.6610  (1.6s)
  fold 5: val_AUC=0.6023  (2.1s)
  → fold AUCs: ['0.6631', '0.6026', '0.6622', '0.6610', '0.6023']
  → mean fold AUC: 0.6382  (std 0.0292)
  → OOF AUC (training): 0.5568
  → Test AUC (hold-out, ensemble): 0.6830

=== M_A_gbm ===


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 1: val_AUC=0.5701  (1.4s)


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 2: val_AUC=0.5938  (1.4s)


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 3: val_AUC=0.5968  (1.4s)


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  fold 4: val_AUC=0.6146  (1.5s)
  fold 5: val_AUC=0.5697  (1.5s)
  → fold AUCs: ['0.5701', '0.5938', '0.5968', '0.6146', '0.5697']
  → mean fold AUC: 0.5890  (std 0.0171)
  → OOF AUC (training): 0.5874
  → Test AUC (hold-out, ensemble): 0.6032


c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 7. Resumen comparativo de las 6 configuraciones

Reportamos las tres métricas. La comparación honesta es **AUC test ensemble** (es la que generaliza fuera del training pool). Las otras dos ayudan a entender el comportamiento dentro de training.

In [8]:
rows = []
for name, r in results.items():
    rows.append({
        'config'        : name,
        'oof_auc'       : r['oof_auc'],
        'mean_fold_auc' : r['mean_fold_auc'],
        'std_fold_auc'  : r['std_fold_auc'],
        'test_auc'      : r['test_auc'],
    })
df_res = pd.DataFrame(rows)
print(df_res.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
df_res.to_csv(os.path.join(PRED_DIR, 'resumen_aucs.csv'), index=False)
print(f'\nGuardado en {PRED_DIR}\\resumen_aucs.csv')

 config  oof_auc  mean_fold_auc  std_fold_auc  test_auc
E_A_mlp   0.5964         0.6522        0.0403    0.6211
E_A_gbm   0.5787         0.5809        0.0270    0.5953
E_B_mlp   0.6247         0.6544        0.0367    0.6065
E_B_gbm   0.6213         0.6313        0.0143    0.6312
M_A_mlp   0.5568         0.6382        0.0292    0.6830
M_A_gbm   0.5874         0.5890        0.0171    0.6032

Guardado en C:\Users\victo\Documents\TFM\Proyecto\Outputs\Predicciones\resumen_aucs.csv


## Siguiente paso

`06_evaluacion.ipynb` — evaluación detallada: AUC con IC 95% bootstrap, Brier, ECE, curvas ROC, y test pareado de DeLong dentro de cada granularidad para ver qué diferencias son estadísticamente significativas.